# 01 — Process DIABIMMUNE 16S data (R)

This notebook keeps genus-level taxa, gives unresolved genus-level taxa unique serialized names, performs dataset-level cleanup, and saves only generated data under
`data/processed_16s/<timestamp>/`.

No plots are created here. All figures are created by notebooks 06 and 07 and are
kept separate from generated tables.


In [1]:
COUNTS_FILE <- "../data/DIABIMMUNE/diabimmune_data_16s.csv"
METADATA_FILE <- "../data/DIABIMMUNE/diabimmune_metadata.csv"

SAMPLE_COLUMN <- "SampleID"
SUBJECT_COLUMN <- "subjectID"
TIME_COLUMN <- "age_at_collection"
LABEL_COLUMN <- "country"
MIN_SAMPLES_PER_SUBJECT <- 3


In [2]:
root <- if (dir.exists("data")) "." else ".."
counts_raw <- read.csv(COUNTS_FILE, check.names = FALSE)
metadata <- read.csv(METADATA_FILE, check.names = FALSE)

metadata$sample_id <- as.character(metadata[[SAMPLE_COLUMN]])
metadata$subject_id <- as.character(metadata[[SUBJECT_COLUMN]])
metadata$time <- as.numeric(metadata[[TIME_COLUMN]])
metadata$label <- as.character(metadata[[LABEL_COLUMN]])

rownames(counts_raw) <- as.character(counts_raw[[1]])
counts <- counts_raw[-1]
counts[] <- lapply(counts, as.numeric)

# Keep genus-level features only.
genus <- grepl("\\|g__[^|]*$", colnames(counts))
counts <- counts[, genus, drop = FALSE]

# Give each unresolved genus a unique name so separate taxa never collapse into one "Unknown" genus.
original_features <- colnames(counts)
unknown <- grepl("\\|g__$", original_features)
serialized_features <- original_features
serialized_features[unknown] <- paste0(
  sub("\\|g__$", "", original_features[unknown]),
  "|g__Unknown_", sprintf("%03d", seq_len(sum(unknown)))
)
colnames(counts) <- serialized_features
feature_map <- data.frame(original_feature = original_features, feature = serialized_features)

metadata <- metadata[complete.cases(metadata[c("sample_id", "subject_id", "time", "label")]), ]
metadata <- metadata[metadata$sample_id %in% rownames(counts), ]
counts <- counts[metadata$sample_id, , drop = FALSE]

metadata$library_size <- rowSums(counts)
keep <- metadata$library_size > 0
metadata <- metadata[keep, ]
counts <- counts[keep, , drop = FALSE]

o <- order(-metadata$library_size)
metadata <- metadata[o, ]
counts <- counts[o, , drop = FALSE]
keep <- !duplicated(metadata[c("subject_id", "time")])
metadata <- metadata[keep, ]
counts <- counts[keep, , drop = FALSE]

visits <- table(metadata$subject_id)
keep <- metadata$subject_id %in% names(visits[visits >= MIN_SAMPLES_PER_SUBJECT])
metadata <- metadata[keep, ]
counts <- counts[keep, , drop = FALSE]

o <- order(metadata$subject_id, metadata$time)
metadata <- metadata[o, ]
counts <- counts[o, colSums(counts) > 0, drop = FALSE]
rownames(counts) <- metadata$sample_id
row.names(metadata) <- NULL

output <- file.path(root, "data", "processed_16s", format(Sys.time(), "%Y%m%d_%H%M%S"))
dir.create(output, recursive = TRUE)
write.csv(data.frame(sample_id = rownames(counts), counts, check.names = FALSE),
          file.path(output, "counts.csv"), row.names = FALSE)
write.csv(metadata, file.path(output, "metadata.csv"), row.names = FALSE)
write.csv(feature_map, file.path(output, "genus_feature_map.csv"), row.names = FALSE)

cat("Saved:", output, "\n")
cat(nrow(metadata), "samples,", length(unique(metadata$subject_id)), "subjects,", ncol(counts), "genus features\n")
cat(sum(unknown), "unresolved genera serialized as g__Unknown_001, g__Unknown_002, ...\n")


Saved: ../data/processed_16s/20260806_210503 
1564 samples, 209 subjects, 282 features
